# Retail Company Web Analysis with Perplexity (via SAP AI Core)

End-to-end workflow: Live orchestration with Perplexity sonar-pro → Evaluation → Optimization

## 1. Setup

In [7]:
%pip install httpx boto3 python-dotenv generative-ai-hub-sdk --quiet


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
import json, os, time
from pathlib import Path
from urllib.parse import quote
import httpx, boto3
from dotenv import load_dotenv
from gen_ai_hub.orchestration_v2.models.config import ModuleConfig, OrchestrationConfig
from gen_ai_hub.orchestration_v2.models.llm_model_details import LLMModelDetails
from gen_ai_hub.orchestration_v2.models.message import SystemMessage, UserMessage
from gen_ai_hub.orchestration_v2.models.template import PromptTemplatingModuleConfig, Template
from gen_ai_hub.orchestration_v2.service import OrchestrationService

load_dotenv(override=True)

# =============================================================================
# Configuration Constants
# =============================================================================
USE_CASE_NAME = "retail-company-web-analysis"
RUN_ID = time.strftime("%Y%m%d-%H%M%S")
LOCAL_WORKDIR = Path("generated") / RUN_ID
LOCAL_WORKDIR.mkdir(parents=True, exist_ok=True)

# Model for live orchestration (Perplexity via AI Core for web search)
MODEL_NAME = "sonar-pro"

# AI Core settings
AICORE_RESOURCE_GROUP = "default"

# Optimization target model
# NOTE: Perplexity models are NOT supported for prompt optimization
# Using GPT-4o instead (supported models: gpt-4o:2024-08-06, gemini-2.5-pro:001)
TARGET_MODEL = "gpt-4o:2024-08-06"

# =============================================================================
# Service Credentials (from .env)
# =============================================================================
AICORE_AUTH_URL = os.environ["AICORE_AUTH_URL"]
AICORE_CLIENT_ID = os.environ["AICORE_CLIENT_ID"]
AICORE_CLIENT_SECRET = os.environ["AICORE_CLIENT_SECRET"]
AICORE_BASE_URL = os.environ["AICORE_BASE_URL"].rstrip("/") + "/v2"

AWS_ACCESS_KEY_ID = os.environ["AWS_ACCESS_KEY_ID"]
AWS_SECRET_ACCESS_KEY = os.environ["AWS_SECRET_ACCESS_KEY"]
AWS_REGION = os.environ["AWS_REGION"]
AWS_S3_BUCKET = os.environ["AWS_S3_BUCKET"]

print(f"Run ID: {RUN_ID}")
print(f"Model for orchestration: {MODEL_NAME}")
print(f"Model for optimization: {TARGET_MODEL}")

Run ID: 20260312-110503
Model for orchestration: sonar-pro
Model for optimization: gpt-4o:2024-08-06


## 2. Prompt Template & Dataset

In [9]:
# =============================================================================
# PROMPT TEMPLATE - Intentionally basic/weak for optimization demonstration
# =============================================================================
# The initial prompt is deliberately vague and unstructured so the optimizer
# can show meaningful improvement by learning from the golden dataset examples.

PROMPT_SPEC = {
    "template": [
        {"role": "system", "content": "You are an assistant. Answer questions about companies."},
        {"role": "user", "content": "Tell me about {{?company_name}}"}
    ]
}

# =============================================================================
# GOLDEN DATASET - High-quality structured examples for the optimizer to learn from
# =============================================================================
# These examples demonstrate the IDEAL output format:
# - Structured with clear sections (Background, Market Position, Products, News, Competition)
# - Factual and specific (numbers, dates, percentages)
# - Concise but comprehensive
# - Professional tone suitable for business stakeholders

DATASET = [
    {
        "fields": {"company_name": "Walmart"},
        "answer": """## Company Overview: Walmart

**Background:** Walmart Inc., founded by Sam Walton in 1962, is headquartered in Bentonville, Arkansas. It is the world's largest retailer by revenue, operating over 10,500 stores across 24 countries.

**Market Position:** Walmart holds approximately 25% of the U.S. grocery market share and generates over $600 billion in annual revenue. The company employs 2.1 million associates worldwide.

**Key Products & Services:** Core offerings include groceries, general merchandise, pharmacy, and financial services. Strategic initiatives include Walmart+, curbside pickup, and same-day delivery through Spark Driver network.

**Recent Developments:** Investments in supply chain automation, healthcare clinics expansion, and advertising business (Walmart Connect) generating $3+ billion annually.

**Competitive Landscape:** Primary competitors are Amazon, Target, and Costco. Walmart differentiates through price leadership, physical store density, and omnichannel integration."""
    },
    {
        "fields": {"company_name": "Target"},
        "answer": """## Company Overview: Target

**Background:** Target Corporation, founded in 1902 as Dayton's and rebranded in 1962, is headquartered in Minneapolis, Minnesota. It operates approximately 1,950 stores across all 50 U.S. states.

**Market Position:** Target is the 8th largest U.S. retailer with annual revenue exceeding $100 billion. Known for "cheap chic" positioning, blending value with style.

**Key Products & Services:** Curated merchandise including exclusive designer collaborations, private labels (Good & Gather, All in Motion), grocery, electronics, and home goods. Services include Drive Up, Shipt same-day delivery, and Target Circle loyalty program.

**Recent Developments:** Store remodels focusing on expanded fulfillment capabilities, sortation centers for faster delivery, and expansion of in-store Ulta Beauty and Apple shop-in-shops.

**Competitive Landscape:** Competes directly with Walmart on value and Amazon on convenience. Differentiates through design partnerships and curated shopping experience."""
    },
    {
        "fields": {"company_name": "Costco"},
        "answer": """## Company Overview: Costco

**Background:** Costco Wholesale Corporation, founded in 1983 through merger of Price Club and Costco, is headquartered in Issaquah, Washington. Operates 850+ warehouse locations globally.

**Market Position:** World's 3rd largest retailer and #1 membership warehouse club. Annual revenue exceeds $230 billion with 120+ million cardholders worldwide. Industry-leading membership renewal rate of 93%.

**Key Products & Services:** Bulk merchandise, Kirkland Signature private label ($60+ billion annually), optical, pharmacy, travel services, and gas stations. Known for limited SKU strategy (4,000 vs 30,000+ at competitors).

**Recent Developments:** E-commerce expansion, international growth particularly in Asia, gold bar sales popularity, and wage leadership (minimum $18.50/hour).

**Competitive Landscape:** Primary competitor is Sam's Club (Walmart). Differentiates through product quality, employee treatment, and treasure-hunt shopping experience."""
    },
    {
        "fields": {"company_name": "Amazon"},
        "answer": """## Company Overview: Amazon

**Background:** Amazon.com, Inc., founded by Jeff Bezos in 1994 as online bookstore, is headquartered in Seattle, Washington. Now a diversified technology and retail conglomerate.

**Market Position:** Largest e-commerce company globally with ~40% U.S. online retail market share. Total revenue exceeds $570 billion. AWS is the leading cloud provider with 32% market share.

**Key Products & Services:** E-commerce marketplace, Prime membership (200+ million subscribers), AWS cloud services, Alexa/Echo devices, Prime Video streaming, Whole Foods, Amazon Fresh, and advertising ($40+ billion business).

**Recent Developments:** AI integration across services, drone delivery expansion, healthcare ventures (One Medical acquisition), and Project Kuiper satellite internet.

**Competitive Landscape:** Competes with Walmart (retail), Microsoft/Google (cloud), Netflix (streaming). Faces antitrust scrutiny and union organizing efforts."""
    },
    {
        "fields": {"company_name": "The Home Depot"},
        "answer": """## Company Overview: The Home Depot

**Background:** The Home Depot, Inc., founded in 1978 by Bernie Marcus and Arthur Blank, is headquartered in Atlanta, Georgia. World's largest home improvement retailer.

**Market Position:** Operates 2,300+ stores across North America. Annual revenue exceeds $150 billion. Holds approximately 30% of the home improvement retail market.

**Key Products & Services:** Building materials, home improvement products, lawn/garden, tools, and appliances. Pro sales (professional contractors) represent 50% of revenue. Services include tool rental, installation, and design.

**Recent Developments:** Supply chain investments (flatbed distribution centers), Pro ecosystem enhancement, interconnected retail experience, and acquisition of SRS Distribution for $18 billion.

**Competitive Landscape:** Primary competitor is Lowe's. Amazon emerging in categories like tools and building materials. Benefits from aging housing stock driving renovation demand."""
    },
    {
        "fields": {"company_name": "Lowe's"},
        "answer": """## Company Overview: Lowe's

**Background:** Lowe's Companies, Inc., founded in 1946 in North Wilkesboro, North Carolina, is headquartered in Mooresville, North Carolina. Second-largest home improvement retailer globally.

**Market Position:** Operates approximately 1,700 stores in the U.S. and Canada. Annual revenue around $90 billion. Focus on both DIY consumers and Pro customers.

**Key Products & Services:** Home improvement products, appliances, lawn/garden, tools, and building materials. MyLowe's Rewards loyalty program, installation services, and Pro loyalty program.

**Recent Developments:** Digital transformation under CEO Marvin Ellison, supply chain modernization, store productivity improvements, and exited international markets to focus on North America.

**Competitive Landscape:** Primary competitor is The Home Depot (significantly larger). Differentiating through store experience, customer service, and rural/suburban market focus."""
    },
    {
        "fields": {"company_name": "Best Buy"},
        "answer": """## Company Overview: Best Buy

**Background:** Best Buy Co., Inc., founded in 1966 as Sound of Music, is headquartered in Richfield, Minnesota. Largest consumer electronics retailer in the United States.

**Market Position:** Operates approximately 1,000 stores across North America. Annual revenue around $45 billion. Successfully navigated "retail apocalypse" through transformation strategy.

**Key Products & Services:** Consumer electronics, appliances, computing, mobile phones, and entertainment. Geek Squad services, Totaltech membership ($200/year for unlimited support), and in-home consultation.

**Recent Developments:** Health technology focus (senior care monitoring), EV charging installations, store-within-store concepts, and outlet store expansion for refurbished products.

**Competitive Landscape:** Competes with Amazon (price), Walmart (selection), and specialty retailers. Differentiates through expert advice, services, and hands-on product experience."""
    },
    {
        "fields": {"company_name": "Kroger"},
        "answer": """## Company Overview: Kroger

**Background:** The Kroger Co., founded in 1883 by Bernard Kroger, is headquartered in Cincinnati, Ohio. Largest supermarket chain by revenue in the United States.

**Market Position:** Operates nearly 2,800 stores under multiple banners (Ralphs, Fred Meyer, Harris Teeter, King Soopers). Annual revenue exceeds $145 billion. Employs 420,000 associates.

**Key Products & Services:** Grocery retail, pharmacy, fuel centers, and Our Brands private label portfolio. Kroger Boost delivery membership and digital coupon program with 60+ million users.

**Recent Developments:** Proposed $25 billion merger with Albertsons (facing regulatory challenges), Ocado automated fulfillment center partnership, and alternative profit streams (media, personal finance).

**Competitive Landscape:** Competes with Walmart (price), Amazon/Whole Foods (digital), and regional grocers. Differentiation through loyalty program data and store brands."""
    },
    {
        "fields": {"company_name": "Walgreens"},
        "answer": """## Company Overview: Walgreens

**Background:** Walgreens Boots Alliance, Inc., founded in 1901, is headquartered in Deerfield, Illinois. One of the largest pharmacy chains in the United States.

**Market Position:** Operates approximately 8,500 stores in the U.S. plus international presence through Boots (UK). Annual revenue around $140 billion. Facing significant transformation challenges.

**Key Products & Services:** Pharmacy services (70% of revenue), front-end retail, healthcare clinics, and photo services. myWalgreens loyalty program with 100+ million members.

**Recent Developments:** VillageMD primary care clinic rollout (1,000+ locations), store closures to optimize footprint, cost reduction initiatives, and exploring strategic alternatives.

**Competitive Landscape:** Primary competitor is CVS Health. Amazon Pharmacy emerging threat. Facing reimbursement pressures and declining front-store traffic."""
    },
    {
        "fields": {"company_name": "CVS Health"},
        "answer": """## Company Overview: CVS Health

**Background:** CVS Health Corporation, founded in 1963, is headquartered in Woonsocket, Rhode Island. Diversified healthcare company and largest pharmacy chain by prescription volume.

**Market Position:** Operates nearly 10,000 retail locations and 1,100+ MinuteClinic locations. Annual revenue exceeds $350 billion. Owns Aetna insurance (acquired 2018 for $69 billion).

**Key Products & Services:** Pharmacy services, retail, health insurance (Aetna), pharmacy benefits management (Caremark), and healthcare clinics. ExtraCare loyalty program.

**Recent Developments:** Health hub store format transformation, Signify Health acquisition for home health, Oak Street Health acquisition for primary care, and virtual care expansion.

**Competitive Landscape:** Competes with Walgreens (pharmacy), UnitedHealth (integrated care), and Amazon. Differentiates through vertical integration across healthcare value chain."""
    },
    {
        "fields": {"company_name": "TJX Companies"},
        "answer": """## Company Overview: TJX Companies

**Background:** The TJX Companies, Inc., founded in 1976, is headquartered in Framingham, Massachusetts. World's largest off-price retailer.

**Market Position:** Operates 4,800+ stores globally under T.J. Maxx, Marshalls, HomeGoods, Winners (Canada), and TK Maxx (Europe). Annual revenue exceeds $52 billion.

**Key Products & Services:** Off-price apparel, accessories, home goods, and beauty products. "Treasure hunt" shopping experience with rapidly rotating inventory. TJX Rewards credit program.

**Recent Developments:** HomeGoods expansion, international growth in Australia and Europe, supply chain investments, and digital enhancement while maintaining store-first strategy.

**Competitive Landscape:** Primary competitors are Ross Stores and Burlington. Differentiates through buying scale (21,000+ vendors), brand relationships, and store locations."""
    },
    {
        "fields": {"company_name": "Dollar General"},
        "answer": """## Company Overview: Dollar General

**Background:** Dollar General Corporation, founded in 1939, is headquartered in Goodlettsville, Tennessee. One of the fastest-growing retailers in America.

**Market Position:** Operates 19,000+ stores across 47 states, primarily in rural and suburban areas. Annual revenue around $38 billion. Opens approximately 1,000 new stores annually.

**Key Products & Services:** Consumables (80% of sales), seasonal items, home products, and apparel. DG Fresh refrigerated/frozen foods program and DG Go! checkout app.

**Recent Developments:** Popshelf banner for higher-income consumers, pOpshelf shop-in-shops within Dollar General, distribution center investments, and health/beauty expansion.

**Competitive Landscape:** Competes with Dollar Tree/Family Dollar, Walmart, and regional grocers. Differentiates through small-box convenience in underserved markets."""
    },
    {
        "fields": {"company_name": "Macy's"},
        "answer": """## Company Overview: Macy's

**Background:** Macy's, Inc., founded in 1858, is headquartered in New York City. Iconic American department store operator facing significant transformation.

**Market Position:** Operates approximately 500 stores under Macy's, Bloomingdale's, and Bluemercury banners. Annual revenue around $24 billion. Famous for Thanksgiving Day Parade and Herald Square flagship.

**Key Products & Services:** Apparel, accessories, cosmetics, home furnishings, and gifts. Star Rewards loyalty program, private brands, and marketplace platform for third-party sellers.

**Recent Developments:** "Bold New Chapter" strategy including 150 store closures, focus on top 350 locations, small-format Market by Macy's and Bloomie's stores, and luxury positioning enhancement.

**Competitive Landscape:** Competes with Nordstrom, Kohl's, and online retailers. Challenged by department store decline and shifting consumer preferences."""
    },
    {
        "fields": {"company_name": "Nordstrom"},
        "answer": """## Company Overview: Nordstrom

**Background:** Nordstrom, Inc., founded in 1901 as a shoe store, is headquartered in Seattle, Washington. Premium department store known for exceptional customer service.

**Market Position:** Operates approximately 350 stores including full-line department stores, Nordstrom Rack off-price, and Nordstrom Local service hubs. Annual revenue around $15 billion.

**Key Products & Services:** Premium apparel, shoes, accessories, and cosmetics. Anniversary Sale flagship event, alterations, styling services, and NORDY CLUB loyalty program.

**Recent Developments:** Nordstrom family take-private bid, marketplace expansion, Nordstrom Media Network advertising business, and supply chain investments.

**Competitive Landscape:** Competes with Macy's Bloomingdale's, Saks Fifth Avenue, and luxury e-commerce. Differentiates through service culture and curated brand assortment."""
    },
    {
        "fields": {"company_name": "IKEA"},
        "answer": """## Company Overview: IKEA

**Background:** IKEA, founded in 1943 by Ingvar Kamprad in Sweden, operates through Ingka Holding. World's largest furniture retailer with unique flat-pack, self-assembly model.

**Market Position:** Operates 460+ stores in 60+ countries. Annual revenue exceeds $45 billion (Ingka Group). Known for affordable, Scandinavian-designed furniture and iconic showroom experience.

**Key Products & Services:** Furniture, home accessories, kitchens, and Swedish food market/restaurant. IKEA Family loyalty program, planning tools, and TaskRabbit assembly services (acquired 2017).

**Recent Developments:** Smaller urban format stores (Planning Studios), e-commerce acceleration, circular economy initiatives (furniture buyback), and smart home products expansion.

**Competitive Landscape:** Competes with Wayfair (online), Ashley, and regional furniture retailers. Differentiates through design, price, and immersive shopping experience."""
    },
    {
        "fields": {"company_name": "Aldi"},
        "answer": """## Company Overview: Aldi

**Background:** Aldi, founded in 1946 in Germany, operates as two separate entities: Aldi Nord and Aldi Süd. Aldi Süd operates U.S. stores. Pioneer of limited-assortment discount grocery.

**Market Position:** Operates 2,300+ U.S. stores (Aldi Süd) with plans for 2,400 by 2025. Global footprint includes 10,000+ stores. One of fastest-growing grocers in America.

**Key Products & Services:** Limited assortment of approximately 1,400 SKUs (vs 30,000+ at conventional grocers), 90%+ private label products, and focus on everyday essentials. ALDI Finds rotating specialty items.

**Recent Developments:** Store remodels with expanded fresh and organic offerings, curbside pickup rollout, delivery partnership with Instacart, and wine/beer selection expansion.

**Competitive Landscape:** Competes with Lidl, Walmart, and traditional grocers. Differentiates through extreme efficiency, limited SKUs, and everyday low prices without promotions."""
    },
    {
        "fields": {"company_name": "Sephora"},
        "answer": """## Company Overview: Sephora

**Background:** Sephora, founded in 1969 in France, is owned by LVMH since 1997. Headquartered in Paris. Pioneer of open-sell beauty retail concept.

**Market Position:** Operates 2,700+ stores worldwide with 500+ in North America. Also operates 900+ Sephora at Kohl's shop-in-shops. Leading specialty beauty retailer.

**Key Products & Services:** Prestige cosmetics, skincare, fragrance, and haircare. Beauty Insider loyalty program (25+ million members), Sephora Collection private label, and in-store services (makeovers, skincare consultations).

**Recent Developments:** Kohl's partnership expansion, clean beauty focus (Clean at Sephora), inclusive beauty initiatives, and AI-powered virtual try-on technology.

**Competitive Landscape:** Primary competitor is Ulta Beauty. Also competes with department store beauty counters and Amazon. Differentiates through prestige brand exclusivity and experience."""
    },
    {
        "fields": {"company_name": "Ulta Beauty"},
        "answer": """## Company Overview: Ulta Beauty

**Background:** Ulta Beauty, Inc., founded in 1990, is headquartered in Bolingbrook, Illinois. Largest beauty retailer in the United States by store count.

**Market Position:** Operates approximately 1,400 stores across all 50 states. Annual revenue exceeds $10 billion. Unique positioning offering both prestige and mass-market beauty.

**Key Products & Services:** Cosmetics, skincare, haircare, and fragrance across all price points. In-store salons (hair, skin, brows), Ultamate Rewards loyalty program (40+ million members), and Target shop-in-shops (500+ locations).

**Recent Developments:** Target partnership expansion, Sparked accelerator for emerging brands, UB Media advertising network, and digital services enhancement (GLAMlab virtual try-on).

**Competitive Landscape:** Primary competitor is Sephora. Differentiates through mass + prestige assortment, salon services, and loyalty program generosity."""
    },
    {
        "fields": {"company_name": "Whole Foods Market"},
        "answer": """## Company Overview: Whole Foods Market

**Background:** Whole Foods Market, founded in 1980 in Austin, Texas, acquired by Amazon in 2017 for $13.7 billion. Pioneer of natural and organic grocery retail.

**Market Position:** Operates approximately 530 stores in U.S., Canada, and UK. Premium positioning with strict quality standards. Serves as Amazon's physical grocery flagship.

**Key Products & Services:** Natural and organic groceries, prepared foods, and specialty items. 365 by Whole Foods value brand, Amazon Prime member discounts (10% off sales items, special deals).

**Recent Developments:** Amazon integration (Prime benefits, Amazon lockers, Just Walk Out technology), Daily Shop smaller format, and supplier diversity initiatives.

**Competitive Landscape:** Competes with Sprouts, Trader Joe's, and conventional grocers expanding organic. Challenged by organic price deflation as mainstream retailers match offerings."""
    },
    {
        "fields": {"company_name": "Trader Joe's"},
        "answer": """## Company Overview: Trader Joe's

**Background:** Trader Joe's, founded in 1967 in Pasadena, California, owned by Aldi Nord since 1979. Cult-favorite specialty grocery chain with unique product offerings.

**Market Position:** Operates 560+ stores across 43 states. Estimated annual revenue of $16+ billion. Known for loyal customer base and limited advertising.

**Key Products & Services:** Private-label focus (80%+ of products) with unique, curated selection (~4,000 SKUs). Famous items include Mandarin Orange Chicken, Everything But The Bagel seasoning, and Two Buck Chuck wine.

**Recent Developments:** Cautious expansion (15-20 new stores annually), maintains no e-commerce/delivery strategy, new store format experiments, and continued product innovation.

**Competitive Landscape:** Indirect competition with Whole Foods, Aldi, and conventional grocers. Differentiates through product uniqueness, value, and neighborhood store experience."""
    },
    {
        "fields": {"company_name": "Publix"},
        "answer": """## Company Overview: Publix

**Background:** Publix Super Markets, Inc., founded in 1930 by George Jenkins, is headquartered in Lakeland, Florida. Largest employee-owned company in the United States.

**Market Position:** Operates 1,350+ stores across Southeast U.S. (Florida, Georgia, Alabama, Tennessee, South Carolina, North Carolina, Virginia). Annual revenue exceeds $55 billion. Consistently top-ranked for customer satisfaction.

**Key Products & Services:** Full-service grocery, pharmacy, and famous deli (Pub Subs). Publix brand private label, Aprons Simple Meals, and GreenWise organic line.

**Recent Developments:** Geographic expansion into Kentucky, delivery partnerships (Instacart), pharmacy services enhancement, and continued employee ownership commitment.

**Competitive Landscape:** Competes with Kroger, Walmart, and Aldi in Southeast markets. Differentiates through customer service, store cleanliness, and employee culture."""
    },
    {
        "fields": {"company_name": "Ross Stores"},
        "answer": """## Company Overview: Ross Stores

**Background:** Ross Stores, Inc., founded in 1982, is headquartered in Dublin, California. Second-largest off-price retailer in the United States.

**Market Position:** Operates 2,100+ stores under Ross Dress for Less and dd's DISCOUNTS banners across 41 states. Annual revenue exceeds $20 billion.

**Key Products & Services:** Off-price apparel, footwear, accessories, and home goods at 20-60% below department store prices. No-frills store environment emphasizing value.

**Recent Developments:** Continued store expansion (100+ new stores annually), Midwest and Northeast market penetration, and distribution center investments.

**Competitive Landscape:** Primary competitor is TJX Companies (larger). Also competes with Burlington and department stores. Differentiates through even lower prices and disciplined operations."""
    },
    {
        "fields": {"company_name": "Dick's Sporting Goods"},
        "answer": """## Company Overview: Dick's Sporting Goods

**Background:** Dick's Sporting Goods, Inc., founded in 1948 by Richard Stack, is headquartered in Coraopolis, Pennsylvania. Largest sporting goods retailer in the United States.

**Market Position:** Operates approximately 900 stores including Dick's, Golf Galaxy, Public Lands, and House of Sport experiential concept. Annual revenue exceeds $12 billion.

**Key Products & Services:** Athletic apparel, footwear, equipment, and outdoor gear. Vertical brands (CALIA, DSG, VRST), and services (equipment customization, golf fitting, outdoor adventures).

**Recent Developments:** House of Sport experiential stores (climbing walls, batting cages), Going Going Gone clearance concept, and gun sales restrictions post-Parkland.

**Competitive Landscape:** Competes with specialty retailers, Amazon, and direct-to-consumer brands (Nike, Lululemon). Survived industry consolidation that eliminated Sports Authority."""
    },
    {
        "fields": {"company_name": "Lululemon"},
        "answer": """## Company Overview: Lululemon

**Background:** Lululemon Athletica Inc., founded in 1998 in Vancouver, Canada. Premium athletic apparel brand that created the modern athleisure category.

**Market Position:** Operates 650+ stores globally. Annual revenue exceeds $9 billion. Premium positioning with leggings averaging $100+. Strong brand loyalty and community focus.

**Key Products & Services:** Technical athletic apparel for yoga, running, and training. Men's category growing rapidly. Mirror home fitness (acquired 2020 for $500M), and membership program.

**Recent Developments:** Men's and international expansion, footwear launch, Mirror pivot from hardware to content, and innovation in fabric technology (like Everlux, Nulu).

**Competitive Landscape:** Competes with Nike, Athleta (Gap), Alo Yoga, and Vuori. Differentiates through quality, community (store ambassadors, events), and brand prestige."""
    },
    {
        "fields": {"company_name": "Williams-Sonoma"},
        "answer": """## Company Overview: Williams-Sonoma

**Background:** Williams-Sonoma, Inc., founded in 1956, is headquartered in San Francisco, California. Premium home furnishings and housewares retailer.

**Market Position:** Operates multiple brands: Williams Sonoma, Pottery Barn, Pottery Barn Kids/Teen, West Elm, and Rejuvenation. Annual revenue exceeds $7 billion. Industry-leading e-commerce (65%+ of sales).

**Key Products & Services:** Cookware, home furnishings, furniture, and décor. Design services, gift registry, and B2B hospitality division. Strong private label penetration.

**Recent Developments:** Digital-first strategy acceleration, sustainability commitments (Fair Trade, responsible sourcing), and small-format store optimization.

**Competitive Landscape:** Competes with Crate & Barrel (Euromarket Designs), RH (Restoration Hardware), Wayfair, and Amazon. Differentiates through brand portfolio and design expertise."""
    },
    {
        "fields": {"company_name": "Gap Inc."},
        "answer": """## Company Overview: Gap Inc.

**Background:** Gap Inc., founded in 1969 by Don and Doris Fisher, is headquartered in San Francisco, California. Portfolio includes Old Navy, Gap, Banana Republic, and Athleta.

**Market Position:** Operates 3,400+ stores globally. Annual revenue around $15 billion. Old Navy is largest and most profitable brand. Company has struggled with brand relevance.

**Key Products & Services:** Casual apparel across multiple price points. Old Navy (value family), Gap (casual basics), Banana Republic (workwear), and Athleta (women's athletic).

**Recent Developments:** CEO changes, Gap brand turnaround attempts, Yeezy partnership ended, Old Navy size-inclusive expansion, and cost restructuring initiatives.

**Competitive Landscape:** Competes with H&M, Uniqlo, American Eagle, and fast fashion players. Challenged by direct-to-consumer brands and changing mall dynamics."""
    },
    {
        "fields": {"company_name": "H&M"},
        "answer": """## Company Overview: H&M

**Background:** Hennes & Mauritz AB (H&M), founded in 1947 in Sweden, is headquartered in Stockholm. One of world's largest fashion retailers and fast fashion pioneer.

**Market Position:** Operates 4,400+ stores across 75+ markets under H&M, COS, & Other Stories, Monki, Weekday, and Arket. Annual revenue exceeds €20 billion. Second-largest fashion retailer globally.

**Key Products & Services:** Trend-driven apparel at accessible prices, home goods (H&M Home), and premium offerings (COS, Arket). H&M Move athletic line and designer collaborations.

**Recent Developments:** Sustainability push (garment collecting, Conscious collection), digital acceleration, brand portfolio optimization, and response to Shein competition.

**Competitive Landscape:** Primary competitor is Zara (Inditex). Challenged by ultra-fast fashion (Shein) and sustainability concerns. Differentiates through price accessibility and brand breadth."""
    },
    {
        "fields": {"company_name": "Zara"},
        "answer": """## Company Overview: Zara

**Background:** Zara, founded in 1975 by Amancio Ortega, is the flagship brand of Inditex (world's largest fashion retailer). Headquartered in Arteixo, Spain. Revolutionized fast fashion industry.

**Market Position:** Operates approximately 1,900 Zara stores in 96 markets. Inditex annual revenue exceeds €35 billion with Zara representing majority. Known for 2-week design-to-store cycle.

**Key Products & Services:** Fast fashion apparel for women, men, and children. Zara Home for housewares. Limited inventory strategy creating urgency and reducing markdowns.

**Recent Developments:** Store mega-format concept, sustainability commitments (100% sustainable fabrics by 2025), RFID inventory technology, and integrated online/offline experience.

**Competitive Landscape:** Competes with H&M, Uniqlo, and ultra-fast fashion (Shein). Differentiates through vertical integration, speed, and fashion-forward positioning."""
    },
    {
        "fields": {"company_name": "Wayfair"},
        "answer": """## Company Overview: Wayfair

**Background:** Wayfair Inc., founded in 2002 by Niraj Shah and Steve Conine, is headquartered in Boston, Massachusetts. Leading online destination for home goods and furniture.

**Market Position:** Operates Wayfair, AllModern, Birch Lane, Joss & Main, and Perigold (luxury). Annual revenue around $12 billion. Offers 33+ million products from 23,000+ suppliers.

**Key Products & Services:** Furniture, décor, home improvement, and outdoor products. Drop-ship model with minimal inventory. Wayfair Professional for trade customers, financing options, and design services.

**Recent Developments:** First physical stores (AllModern), significant workforce reductions, path to profitability focus, and advertising business development.

**Competitive Landscape:** Competes with Amazon, IKEA, Williams-Sonoma, and traditional furniture retailers. Challenged by profitability concerns and post-pandemic demand normalization."""
    }
]

# Save datasets
SHARED_DATASET_FILENAME = "retail_company_analysis_dataset.json"
EVAL_DATASET_FILENAME = "retail_company_analysis_eval_dataset.json"
OBJECT_STORE_PATH_PREFIX = f"hackathon/prompt-optimization/{RUN_ID}"

local_dataset_path = LOCAL_WORKDIR / SHARED_DATASET_FILENAME
local_dataset_path.write_text(json.dumps(DATASET, indent=2, ensure_ascii=False), encoding="utf-8")

eval_dataset = [{**item["fields"], "expected_output": item["answer"]} for item in DATASET]
local_eval_path = LOCAL_WORKDIR / EVAL_DATASET_FILENAME
local_eval_path.write_text(json.dumps(eval_dataset, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"✓ Dataset: {len(DATASET)} examples saved")
print(f"\n📌 Initial Prompt (intentionally basic):")
print(f"   System: \"{PROMPT_SPEC['template'][0]['content']}\"")
print(f"   User: \"{PROMPT_SPEC['template'][1]['content']}\"")
print(f"\n📌 Golden examples demonstrate structured format with:")
print("   - Company Overview header")
print("   - Background, Market Position, Products, News, Competition sections")
print("   - Specific facts, numbers, and percentages")

✓ Dataset: 29 examples saved

📌 Initial Prompt (intentionally basic):
   System: "You are an assistant. Answer questions about companies."
   User: "Tell me about {{?company_name}}"

📌 Golden examples demonstrate structured format with:
   - Company Overview header
   - Background, Market Position, Products, News, Competition sections
   - Specific facts, numbers, and percentages


## 3. Live Orchestration Test

In [10]:
template = Template(template=[SystemMessage(content=PROMPT_SPEC["template"][0]["content"]), UserMessage(content=PROMPT_SPEC["template"][1]["content"])])
llm = LLMModelDetails(name=MODEL_NAME, params={"temperature": 0.2, "max_completion_tokens": 1500})
service = OrchestrationService(config=OrchestrationConfig(modules=ModuleConfig(prompt_templating=PromptTemplatingModuleConfig(prompt=template, model=llm))))

# Test with first company from dataset
test_company = DATASET[0]["fields"]
result = service.run(placeholder_values=test_company)
print("Generated Analysis:\n" + result.final_result.choices[0].message.content)

Generated Analysis:
**Walmart** is the world's largest retail company, founded in 1962 by Sam Walton in Rogers, Arkansas[1][2]. The company operates more than 11,500 stores across 28 countries[4] and has grown from a single discount store into a retail empire generating hundreds of billions in annual revenue[6].

## Founding and Early Growth

Sam Walton's retail journey began in 1945 when he purchased a Ben Franklin store, focusing on selling products at low prices to achieve higher-volume sales[2]. He opened the first Walmart store on July 2, 1962, in Rogers, Arkansas, building on his strategy of bringing discount retail to smaller towns rather than urban centers[1][3]. By 1967, the Walton family owned 24 stores with $12.7 million in sales[1].

The company officially incorporated as Wal-Mart Stores, Inc. on October 31, 1969, and became publicly traded on October 1, 1970[2]. By 1970, it had 38 stores, 1,500 employees, and $44.2 million in sales[2]. The company expanded rapidly througho

## 4. AI Core Client

In [11]:
class AICore:
    def __init__(self):
        self._token, self._expiry = None, 0
        self._http = httpx.Client(timeout=120, follow_redirects=True)
    
    def close(self): self._http.close()
    
    def _get_token(self):
        if self._token and time.time() < self._expiry - 60: return self._token
        r = self._http.post(f"{AICORE_AUTH_URL.rstrip('/')}/oauth/token", data={"grant_type": "client_credentials"}, auth=(AICORE_CLIENT_ID, AICORE_CLIENT_SECRET))
        r.raise_for_status()
        self._token, self._expiry = r.json()["access_token"], time.time() + r.json().get("expires_in", 1800)
        return self._token
    
    def req(self, method, path, body=None, params=None, rg=True):
        headers = {"Authorization": f"Bearer {self._get_token()}", "Content-Type": "application/json"}
        if rg: headers["AI-Resource-Group"] = AICORE_RESOURCE_GROUP
        r = self._http.request(method, f"{AICORE_BASE_URL}{path}", json=body, params=params, headers=headers)
        if r.status_code >= 400: raise RuntimeError(f"{method} {path}: {r.status_code}\n{r.text}")
        return r.json() if r.content and "json" in r.headers.get("Content-Type", "") else r.text if r.content else None
    
    def poll(self, getter, obj_id, terminals, timeout=900):
        start = time.time()
        while time.time() - start < timeout:
            p = getter(obj_id)
            status = p.get("status", "UNKNOWN").upper()
            print(f"  {obj_id}: {status}")
            if status in {s.upper() for s in terminals}: return p
            time.sleep(5)
        raise TimeoutError(f"Timeout: {obj_id}")

client = AICore()
print("Client initialized")

Client initialized


## 5. Setup Resources & Upload Data

In [12]:
# =============================================================================
# Prepare Dataset Files for Upload
# =============================================================================
# Dataset filenames (using DATASET from cell 5)
SHARED_DATASET_FILENAME = f"{USE_CASE_NAME}-training-{RUN_ID}.json"
EVAL_DATASET_FILENAME = f"{USE_CASE_NAME}-eval-{RUN_ID}.json"
OBJECT_STORE_PATH_PREFIX = f"prompt-optimizer/{USE_CASE_NAME}/{RUN_ID}"

# Save training dataset locally (using DATASET from cell 5)
local_dataset_path = LOCAL_WORKDIR / SHARED_DATASET_FILENAME
local_dataset_path.write_text(json.dumps(DATASET, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved training dataset: {local_dataset_path} ({len(DATASET)} examples)")

# Save evaluation dataset locally (transform DATASET to eval format)
eval_dataset = [{**item["fields"], "expected_output": item["answer"]} for item in DATASET]
local_eval_path = LOCAL_WORKDIR / EVAL_DATASET_FILENAME
local_eval_path.write_text(json.dumps(eval_dataset, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved eval dataset: {local_eval_path} ({len(eval_dataset)} examples)")

Saved training dataset: generated/20260312-110503/retail-company-web-analysis-training-20260312-110503.json (29 examples)
Saved eval dataset: generated/20260312-110503/retail-company-web-analysis-eval-20260312-110503.json (29 examples)


In [13]:
# Ensure resource group
try: client.req("POST", "/admin/resourceGroups", {"resourceGroupId": AICORE_RESOURCE_GROUP}, rg=False); print(f"Created: {AICORE_RESOURCE_GROUP}")
except: print(f"Exists: {AICORE_RESOURCE_GROUP}")

# Upload datasets to S3
s3 = boto3.client('s3', aws_access_key_id=AWS_ACCESS_KEY_ID, aws_secret_access_key=AWS_SECRET_ACCESS_KEY, region_name=AWS_REGION)
for lp, fn in [(local_dataset_path, SHARED_DATASET_FILENAME), (local_eval_path, EVAL_DATASET_FILENAME)]:
    key = f"{OBJECT_STORE_PATH_PREFIX}/datasets/{fn}"
    s3.put_object(Bucket=AWS_S3_BUCKET, Key=key, Body=lp.read_bytes(), ContentType="application/json")
    print(f"Uploaded: {fn}")

ARTIFACT_ROOT = f"ai://default/{OBJECT_STORE_PATH_PREFIX}/datasets"
print(f"Artifact root: {ARTIFACT_ROOT}")

Exists: default
Uploaded: retail-company-web-analysis-training-20260312-110503.json
Uploaded: retail-company-web-analysis-eval-20260312-110503.json
Artifact root: ai://default/prompt-optimizer/retail-company-web-analysis/20260312-110503/datasets


## 6. Register Prompt Templates

In [14]:
PROMPT_NAME = f"{USE_CASE_NAME}-base-{RUN_ID}"

# For optimizer
opt_prompt = client.req("POST", "/lm/promptTemplates", {"name": PROMPT_NAME, "version": "0.0.1", "scenario": "genai-optimizations", "spec": PROMPT_SPEC})
print(f"Optimizer prompt: genai-optimizations/{PROMPT_NAME}:0.0.1")

# For evaluation
eval_prompt = client.req("POST", "/lm/promptTemplates", {"name": f"{PROMPT_NAME}-eval", "version": "0.0.1", "scenario": "genai-evaluations", "spec": PROMPT_SPEC})
eval_prompt_id = eval_prompt.get("id")
print(f"Eval prompt ID: {eval_prompt_id}")

Optimizer prompt: genai-optimizations/retail-company-web-analysis-base-20260312-110503:0.0.1
Eval prompt ID: 9ee92305-b392-4ee7-96ca-eddf69480e90


## 7. Run Evaluation (Optional)

In [ ]:
# =============================================================================
# EVALUATION METRICS - Uncomment to run evaluation before/after optimization
# =============================================================================
#
# Available metrics for retail company web analysis use case:
#
# RECOMMENDED METRICS:
# - "Pointwise Answer Relevance"  : Measures if the response is relevant to the company query
# - "Pointwise Conciseness"       : Ensures responses are concise and not overly verbose
# - "Pointwise Instruction Following" : Checks if the model follows the analysis structure
# - "Pointwise Correctness"       : Validates factual accuracy of company information

# ADDITIONAL METRICS (for deeper analysis):
# - "groundedness"                : Measures if claims are grounded in retrievable facts
# - "BLEU"                        : N-gram overlap with reference answers (lexical similarity)
# - "ROUGE"                       : Recall-oriented measure for summary quality
# - "BERT Score"                  : Semantic similarity using BERT embeddings

# METRIC SELECTION RATIONALE:
# For web analysis, we prioritize:
# 1. Relevance - Is the analysis about the right company?
# 2. Correctness - Are the facts accurate?
# 3. Conciseness - Is it digestible for stakeholders?
# 4. Instruction Following - Does it cover all required sections?

EVAL_METRICS = "Pointwise Answer Relevance,Pointwise Conciseness,Pointwise Instruction Following,Pointwise Correctness"

# Register artifact
eval_artifact = client.req("POST", "/lm/artifacts", {
    "name": f"{USE_CASE_NAME}-eval-dataset-{RUN_ID}", "kind": "other", "scenarioId": "genai-evaluations",
    "url": ARTIFACT_ROOT, "labels": [{"key": "ext.ai.sap.com/prompt-evaluation", "value": "true"}],
    "description": "Retail company web analysis eval dataset"
})
eval_artifact_id = eval_artifact["id"]

# Create config with comprehensive metrics
eval_config = client.req("POST", "/lm/configurations", {
    "name": f"{USE_CASE_NAME}-eval-{RUN_ID}", "scenarioId": "genai-evaluations", "executableId": "genai-evaluations-simplified",
    "inputArtifactBindings": [{"key": "datasetFolder", "artifactId": eval_artifact_id}],
    "parameterBindings": [
        {"key": "repetitions", "value": "1"},
        {"key": "metrics", "value": EVAL_METRICS},
        {"key": "testDataset", "value": json.dumps({"path": EVAL_DATASET_FILENAME, "type": "json"})},
        {"key": "promptTemplate", "value": eval_prompt_id},
        {"key": "models", "value": "gpt-4o:2024-08-06"},
        {"key": "orchestrationRegistryIds", "value": ""}
    ]
})

# Execute and poll
eval_exec = client.req("POST", "/lm/executions", {"configurationId": eval_config["id"]})
print(f"Evaluation started: {eval_exec['id']}")
client.poll(lambda x: client.req("GET", f"/lm/executions/{x}"), eval_exec["id"], {"COMPLETED", "FAILED", "DEAD"})

# Get and display metrics
try:
    metrics = client.req("GET", "/lm/metrics", params={"tagFilters": f"evaluation.ai.sap.com/child-of={eval_exec['id']}"})
    print("\n" + "=" * 60)
    print("EVALUATION RESULTS")
    print("=" * 60)
    for metric in metrics.get("resources", []):
        print(f"  {metric.get('name', 'N/A')}: {metric.get('value', 'N/A')}")
except Exception as e:
    print(f"Metrics error: {e}")

Evaluation started: e0ff037b5a65a834
  e0ff037b5a65a834: UNKNOWN
  e0ff037b5a65a834: UNKNOWN


## 8. Run Optimization

In [ ]:
# Register artifact
opt_artifact = client.req("POST", "/lm/artifacts", {
    "name": f"{USE_CASE_NAME}-opt-dataset-{RUN_ID}", "kind": "dataset", "scenarioId": "genai-optimizations",
    "url": ARTIFACT_ROOT, "description": "Retail company web analysis optimizer dataset"
})

# =============================================================================
# OPTIMIZATION TARGET MODELS
# =============================================================================
# Supported models for optimization: gpt-4o:2024-08-06, gemini-2.5-pro:001
# Note: Perplexity (sonar-pro) is NOT supported for optimization, only for inference

# Target models configuration
# IMPORTANT: The mapping key must match the model name exactly as it appears in targetModels
# The format is: modelname=promptname:version (without provider prefix in the key)
TARGET_MODELS = [
    {"name": "gpt-4o:2024-08-06", "alias": "gpt4o"},
    {"name": "gemini-2.5-pro:001", "alias": "gemini"}
]

# Build target models string and mapping
# Key insight: The mapping uses the model name as-is (not with provider prefix)
target_models_str = ",".join([m["name"] for m in TARGET_MODELS])
target_prompt_mapping = ",".join([f"{m['name']}={USE_CASE_NAME}-opt-{m['alias']}-{RUN_ID}:0.0.1" for m in TARGET_MODELS])

print(f"Target models: {target_models_str}")
print(f"Prompt mapping: {target_prompt_mapping}")

# Create optimization config
opt_config = client.req("POST", "/lm/configurations", {
    "name": f"{USE_CASE_NAME}-opt-{RUN_ID}", "scenarioId": "genai-optimizations", "executableId": "genai-optimizations",
    "inputArtifactBindings": [{"key": "prompt-data", "artifactId": opt_artifact["id"]}],
    "parameterBindings": [
        {"key": "dataset", "value": SHARED_DATASET_FILENAME}, 
        {"key": "optimizationMetric", "value": "LLMaaJ:Sem_Sim_1"},
        {"key": "basePrompt", "value": f"genai-optimizations/{PROMPT_NAME}:0.0.1"}, 
        {"key": "baseModel", "value": TARGET_MODEL},
        {"key": "targetModels", "value": target_models_str},
        {"key": "targetPromptMapping", "value": target_prompt_mapping}
    ]
})

# Execute and poll
opt_exec = client.req("POST", "/lm/executions", {"configurationId": opt_config["id"]})
opt_exec_id = opt_exec["id"]
print(f"\nOptimization started: {opt_exec_id}")
final_status = client.poll(lambda x: client.req("GET", f"/lm/executions/{x}"), opt_exec_id, {"COMPLETED", "FAILED", "DEAD"})
print(f"Final status: {final_status.get('status', 'UNKNOWN')}")

Target models: gpt-4o:2024-08-06,gemini-2.5-pro:001
Prompt mapping: gpt-4o:2024-08-06=retail-company-web-analysis-opt-gpt4o-20260312-003944:0.0.1,gemini-2.5-pro:001=retail-company-web-analysis-opt-gemini-20260312-003944:0.0.1

Optimization started: e8c688bbcb22b967
  e8c688bbcb22b967: UNKNOWN
  e8c688bbcb22b967: UNKNOWN
  e8c688bbcb22b967: UNKNOWN
  e8c688bbcb22b967: UNKNOWN
  e8c688bbcb22b967: UNKNOWN
  e8c688bbcb22b967: UNKNOWN
  e8c688bbcb22b967: UNKNOWN
  e8c688bbcb22b967: UNKNOWN
  e8c688bbcb22b967: RUNNING
  e8c688bbcb22b967: RUNNING
  e8c688bbcb22b967: RUNNING
  e8c688bbcb22b967: RUNNING
  e8c688bbcb22b967: RUNNING
  e8c688bbcb22b967: RUNNING
  e8c688bbcb22b967: RUNNING
  e8c688bbcb22b967: RUNNING
  e8c688bbcb22b967: RUNNING
  e8c688bbcb22b967: RUNNING
  e8c688bbcb22b967: RUNNING
  e8c688bbcb22b967: RUNNING
  e8c688bbcb22b967: RUNNING
  e8c688bbcb22b967: RUNNING
  e8c688bbcb22b967: RUNNING
  e8c688bbcb22b967: RUNNING
  e8c688bbcb22b967: RUNNING
  e8c688bbcb22b967: RUNNING
  e8c6

## 9. View Results

In [ ]:
# =============================================================================
# OPTIMIZATION RESULTS - Presentation View
# =============================================================================

def display_optimization_results(results):
    """Display optimization results in a clean, presentation-ready format."""
    
    # Header
    print("\n" + "=" * 70)
    print("                    PROMPT OPTIMIZATION RESULTS")
    print("=" * 70)
    
    status = results.get("job_status", "N/A")
    status_icon = "✓" if status == "COMPLETED" else "✗" if status == "FAILED" else "○"
    print(f"\n  Status: {status_icon} {status}")
    print(f"  Use Case: {USE_CASE_NAME}")
    print(f"  Run ID: {RUN_ID}")
    
    # Baseline Model Section
    origin = results.get("origin_model", {})
    if origin:
        print("\n" + "-" * 70)
        print("  BASELINE MODEL (Before Optimization)")
        print("-" * 70)
        print(f"  Model:          {origin.get('model_name', 'N/A')}")
        print(f"  Baseline Score: {origin.get('score', 'N/A')}")
        
        sys_prompt = origin.get("system_prompt", "")
        if sys_prompt:
            print(f"\n  Original System Prompt:")
            print("  " + "─" * 50)
            # Word wrap the prompt for better display
            wrapped = sys_prompt[:300] + ("..." if len(sys_prompt) > 300 else "")
            for line in wrapped.split("\n"):
                print(f"  │ {line}")
            print("  " + "─" * 50)
    
    # Target Models Section (Optimized Results)
    target_models = results.get("target_models", [])
    if target_models:
        print("\n" + "-" * 70)
        print("  OPTIMIZED MODELS (After Optimization)")
        print("-" * 70)
        
        # Summary table
        print("\n  ┌─────────────────────────┬────────────┬────────────┬────────────┐")
        print("  │ Model                   │ Pre-Score  │ Post-Score │ Improvement│")
        print("  ├─────────────────────────┼────────────┼────────────┼────────────┤")
        
        for target in target_models:
            model = target.get("model_name", "N/A")[:23]
            pre = target.get("pre_optimization_score", 0)
            post = target.get("post_optimization_score", 0)
            
            if isinstance(pre, (int, float)) and isinstance(post, (int, float)):
                improvement = post - pre
                imp_str = f"+{improvement:.4f}" if improvement >= 0 else f"{improvement:.4f}"
                pre_str = f"{pre:.4f}"
                post_str = f"{post:.4f}"
            else:
                imp_str = "N/A"
                pre_str = str(pre)[:10]
                post_str = str(post)[:10]
            
            print(f"  │ {model:<23} │ {pre_str:>10} │ {post_str:>10} │ {imp_str:>10} │")
        
        print("  └─────────────────────────┴────────────┴────────────┴────────────┘")
        
        # Detailed optimized prompts
        print("\n  OPTIMIZED PROMPTS:")
        for i, target in enumerate(target_models, 1):
            model = target.get("model_name", "N/A")
            sys_prompt = target.get("system_prompt", "")
            
            print(f"\n  [{i}] {model}")
            print("  " + "═" * 60)
            
            if sys_prompt:
                # Display the full optimized prompt
                print(f"\n{sys_prompt}")
            else:
                print("  (No optimized prompt available)")
            
            print("\n  " + "═" * 60)
    
    # Key Insights
    if target_models:
        print("\n" + "-" * 70)
        print("  KEY INSIGHTS")
        print("-" * 70)
        
        # Find best improvement
        best_model = None
        best_improvement = -float('inf')
        
        for target in target_models:
            pre = target.get("pre_optimization_score", 0)
            post = target.get("post_optimization_score", 0)
            if isinstance(pre, (int, float)) and isinstance(post, (int, float)):
                improvement = post - pre
                if improvement > best_improvement:
                    best_improvement = improvement
                    best_model = target.get("model_name", "N/A")
        
        if best_model:
            print(f"\n  → Best performing model: {best_model}")
            print(f"  → Score improvement: +{best_improvement:.4f}")
            print(f"  → Optimization metric: LLMaaJ:Sem_Sim_1 (Semantic Similarity)")
    
    print("\n" + "=" * 70)
    print("                         END OF RESULTS")
    print("=" * 70 + "\n")


# Fetch and display results
try:
    results_path = f"default/{opt_exec_id}/result-data/results.json"
    raw_results = client.req("GET", f"/lm/dataset/files/{quote(results_path, safe='')}")
    
    # Parse JSON if returned as string
    results = json.loads(raw_results) if isinstance(raw_results, str) else raw_results
    display_optimization_results(results)
    
    # Save results locally for reference
    results_file = LOCAL_WORKDIR / "optimization_results.json"
    results_file.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Results saved to: {results_file}")
    
except Exception as e:
    print(f"\n⚠ Could not fetch results: {e}")
    print("\nPossible reasons:")
    print("  1. Optimization is still running (check status above)")
    print("  2. Optimization failed - check AI Launchpad for logs")
    print("  3. Results file path has changed")

client.close()
print("\n✓ Session complete!")


                    PROMPT OPTIMIZATION RESULTS

  Status: ○ completed
  Use Case: retail-company-web-analysis
  Run ID: 20260312-003944

----------------------------------------------------------------------
  BASELINE MODEL (Before Optimization)
----------------------------------------------------------------------
  Model:          gpt-4o:2024-08-06
  Baseline Score: 0.4

  Original System Prompt:
  ──────────────────────────────────────────────────
  │ You are an assistant. Answer questions about companies.
  ──────────────────────────────────────────────────

----------------------------------------------------------------------
  OPTIMIZED MODELS (After Optimization)
----------------------------------------------------------------------

  ┌─────────────────────────┬────────────┬────────────┬────────────┐
  │ Model                   │ Pre-Score  │ Post-Score │ Improvement│
  ├─────────────────────────┼────────────┼────────────┼────────────┤
  │ gemini-2.5-pro:001      │     0.52